# Embedding and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What the course asks for in Sprint 2:** *"the most basic vector database and retrieval over your documents"*, with the parameters in `config/rag.yaml`, and by 11/11: *"retrieval returns passages for a handful of test questions"*.

**What this notebook does:** it turns our 105 chunks into vectors, stores them, and returns the 5 best chunks for a question. Every choice says where it comes from.

| Step | What | Where the choice comes from |
|---|---|---|
| 1 | Load the settings and the chunks | `config/rag.yaml` (course rule) |
| 2 | Prepare the text: nothing removed by default | Dickinson 2026, *Stop Removing Stopwords* |
| 3 | Which words BM25 already ignores (IDF) | the BM25 formula |
| 4 | A jury of three LLMs labels the frequent words | instructor's feedback; Verga et al. 2024, *Replacing Judges with Juries* |
| 5 | Does removing words help? Tested on my questions | Dickinson 2026: *"the burden of proof sits with removal"* |
| 6 | Embed the chunks with `bge-base-en-v1.5` | design decision D10, model card |
| 7 | Store them in Chroma | design decision D11 |
| 8 | Keyword index (BM25) | design decision D12 |
| 9 | Search: dense (E1) and hybrid (E2) | design decision D12, RRF paper |
| 10 | A handful of test questions (11/11 check) | course halfway check |
| 11 | A picture of the chunks | my earlier retrieval project |

The measurement on the `val` questions (Recall@5, MRR, the instructor's RAG metrics, W&B) is the next task, KAN-99.

In [ ]:
%pip -q install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn umap-learn matplotlib
import csv, json, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml
pd.set_option("display.max_colwidth", None)
np.random.seed(42)

## 1. Load the settings and the chunks
All settings are in `config/rag.yaml`: the course asks for parameters there, not in the code. The chunks come from `src/chunking.ipynb`.

In [ ]:
CONFIG = yaml.safe_load(open("../config/rag.yaml"))
chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
print(len(chunks), "chunks")
CONFIG

## 2. Prepare the text: nothing is removed by default
**Dense search (embedding)** reads the full `embed_text`: heading + the sentence before a point + the provision. The model reads normal sentences, and small words change a rule: *"shall **not** be considered to be high-risk"* means the opposite without "not".

**Keyword search (BM25)** reads the same text in lower case, split into words. We **do not remove stopwords by default**. A 2026 study on court decisions (Dickinson, *Stop Removing Stopwords*) found that the usual stopword lists (scikit-learn, NLTK, spaCy) made legal text classification slightly worse in all 12 tests, and that even word-by-word tuned lists gave no measurable gain. Its rule: *"the burden of proof sits with removal"*: a word is removed only if removing it is shown to help (step 5).

In [ ]:
words = lambda text: re.findall(r"[a-z0-9]+", text.lower())
example = next(c for c in chunks if c["chunk_id"] == "ART6-3-1")
print("TEXT:", example["text"][:150], "...")
print("BM25:", words(example["text"])[:18], "...")
print(sum(any(w in {"not", "no", "nor"} for w in words(c["text"])) for c in chunks), "of 105 chunks contain a negation")

**Comment:** _(write 1–2 sentences after running)_

## 3. Which words does BM25 already ignore?
BM25 gives every word a weight (IDF): a word in many chunks gets a weight close to 0, so it hardly counts in a search. For comparison, a word in only one chunk gets about 4.3. This is part of BM25 itself, so frequent words are already pushed down without any list. The table shows the **20 words with the lowest weight**, how many chunks and questions contain them. These 20 are only the candidates for step 4.

In [ ]:
from rank_bm25 import BM25Okapi
questions = {r["id"]: r for r in csv.DictReader(open("../docs/eval/eval-set-v1.csv", encoding="utf-8-sig"))}
in_q = Counter(w for q in questions.values() for w in set(words(q["question"])))
in_c = Counter(w for c in chunks for w in set(words(c["embed_text"])))

N = len(chunks)
idf = lambda n: np.log(1 + (N - n + 0.5) / (n + 0.5))     # BM25 word weight (as in Lucene / Elasticsearch)
weights = pd.DataFrame([{"word": w, "BM25 weight (IDF)": round(idf(n), 2), "% of chunks": round(100 * n / N),
                         "% of questions": round(100 * in_q[w] / len(questions))} for w, n in in_c.items()])
candidates = weights.sort_values("BM25 weight (IDF)").head(20).reset_index(drop=True)
candidates

**Comment:** _(write 1–2 sentences after running)_

## 4. A jury of three LLMs labels the candidates
The instructor asked us to let an LLM decide which frequent words are useless, with a **different model** than our answer model. Instead of one judge, we use a **jury of three small models from different companies**, and the majority wins. Verga et al. (2024, *Replacing Judges with Juries*) showed that a panel of three models from different families agrees better with people than one large judge (Cohen's kappa 0.76 vs 0.63 for GPT-4 on Natural Questions), has less bias towards its own family, and is cheaper; for yes/no labels they use the majority vote.

| Model | Company | Why |
|---|---|---|
| `qwen3:4b` | Alibaba | worked with thinking turned off in our model selection spike |
| `phi4-mini` | Microsoft | our fallback answer model in the spike (D9) |
| `gemma3:4b` | Google | a third family of the same size |

None is **Llama** (our answer model, D9) or **Claude** (used to draft the questions). Same prompt, three examples of each word from our chunks, temperature 0. Needs `ollama pull qwen3:4b`, `ollama pull phi4-mini`, `ollama pull gemma3:4b`.

In [ ]:
PROMPT = """This word appears in many EU AI Act provisions. Examples:
{examples}
Should a keyword search IGNORE it (common wording that does not help to find the right provision)
or KEEP it (a question can depend on it)? If in doubt, KEEP.
Answer in one line: IGNORE or KEEP, then a reason of at most 12 words.
WORD: {word}"""

def ask(model, prompt):
    """One answer from a model in Ollama, temperature 0 (thinking off for Qwen3)."""
    body = {"model": model, "stream": False, "options": {"temperature": 0}, "messages": [{"role": "user", "content": prompt}]}
    if model.startswith("qwen3"):
        body["think"] = False
    return requests.post(CONFIG["judge"]["url"], json=body).json()["message"]["content"].strip()

def examples(word):
    return "\n".join([f'- {c["ref"]}: {c["text"][:150]}...' for c in chunks if word in words(c["text"])][:3])

jury = candidates[["word", "% of questions"]].copy()
for model in CONFIG["judge"]["jury"]:
    jury[model] = [ask(model, PROMPT.format(examples=examples(w), word=w)) for w in jury["word"]]
votes = jury[CONFIG["judge"]["jury"]].apply(lambda col: col.str.upper().str.startswith("IGNORE"))
jury["majority"] = np.where(votes.sum(axis=1) >= 2, "IGNORE", "KEEP")
JURY_IGNORE = set(jury.loc[jury["majority"] == "IGNORE", "word"])
print("The jury would ignore:", sorted(JURY_IGNORE))
jury

**Check:** read the reasons; does the majority make sense? Comment:** _(write 1–2 sentences after running)_

## 5. Does removing words help? Tested on my questions
Following the *burden of proof* rule, we compare three versions of BM25 on my questions **B-01 to B-06** (Annex III 3(b), planned for the `val` split):
- **none:** nothing removed (the default)
- **jury:** the words the jury marked IGNORE are removed
- **stopwords:** the scikit-learn stopword list is removed, but "not", "no", "nor" are kept

Scores, as in the instructor's slides (*"compute Recall@k / MRR on gold IDs"*): **Recall@5** = share of the gold provisions in the top 5; **MRR** = 1 / rank of the first correct chunk.

**Rule:** a version replaces "none" only if it is better on Recall@5 or MRR and worse on neither. Six questions are few (one question changes Recall@5 by about 0.17), so this is a first check; it is repeated on all the team's `val` questions in KAN-99.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
REMOVE = {"none": set(), "jury": JURY_IGNORE, "stopwords": set(ENGLISH_STOP_WORDS) - {"not", "no", "nor"}}
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
dev = [questions[q] for q in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]]

def bm25_search(remove):
    index = BM25Okapi([[w for w in words(c["embed_text"]) if w not in remove] for c in chunks])
    return lambda q: [chunks[n]["chunk_id"] for n in np.argsort(-index.get_scores([w for w in words(q) if w not in remove]))]

def score(search, k=5):
    """Recall@k and MRR over my questions B-01 to B-06."""
    recall, rr = [], []
    for q in dev:
        gold = [g.strip() for g in q["gold_source"].split(";")]
        refs = [ref_of[c] for c in search(q["question"])[:k]]
        hit = lambda r: any(r.startswith(g) or g.startswith(r) for g in gold)
        recall.append(np.mean([any(r.startswith(g) or g.startswith(r) for r in refs) for g in gold]))
        rr.append(next((1 / n for n, r in enumerate(refs, 1) if hit(r)), 0))
    return {"Recall@5": round(np.mean(recall), 2), "MRR": round(np.mean(rr), 2)}

test = pd.DataFrame([{"version": v, **score(bm25_search(words_out))} for v, words_out in REMOVE.items()]).set_index("version")
better = lambda v: (test.loc[v] >= test.loc["none"]).all() and (test.loc[v] > test.loc["none"]).any()
CHOSEN = next((v for v in ["jury", "stopwords"] if better(v)), "none")
print("Chosen:", CHOSEN, "(write it in config/rag.yaml, bm25: remove)")
test

**Comment:** _(write 1–2 sentences after running)_

## 6. Embed the chunks
`BAAI/bge-base-en-v1.5` (D10, chosen in the model selection spike) turns each text into 768 numbers; texts with a similar meaning get similar numbers. The vectors are normalised, so cosine similarity is a dot product. As its model card recommends, questions get a short instruction in front (`query_instruction` in the config); chunks do not.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(CONFIG["embedding"]["model"], device="cpu")
embed = lambda texts: model.encode(texts, normalize_embeddings=True)

vectors = embed([c["embed_text"] for c in chunks])
print("vectors:", vectors.shape)

i = {c["chunk_id"]: n for n, c in enumerate(chunks)}
print("ART6-3-a vs ART6-3-b (two points of the same rule):", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART6-3-b"]]), 2))
print("ART6-3-a vs ART113-3-c-i (a different topic):      ", round(float(vectors[i["ART6-3-a"]] @ vectors[i["ART113-3-c-i"]]), 2))

Two points of the same rule should be closer (higher number) than two different topics.

**Comment:** _(write 1–2 sentences after running)_

## 7. Store them in Chroma
Chroma (D11) keeps the vectors, the texts and the metadata, and returns the closest chunks with cosine distance. It is saved in `data/vector-store/`; it is not in Git, because this notebook rebuilds it in seconds.

In [ ]:
import chromadb
store = chromadb.PersistentClient(path=CONFIG["vector_store"]["path"])
if "chunks" in [c.name for c in store.list_collections()]:
    store.delete_collection("chunks")
col = store.create_collection("chunks", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in chunks], embeddings=vectors.tolist(), documents=[c["text"] for c in chunks],
        metadatas=[{"ref": c["ref"], "version_date": c["version_date"]} for c in chunks])
print(col.count(), "chunks stored")

## 8. Keyword index (BM25)
BM25 (D12) gives a high score to chunks that share the question's words, especially rare ones. It finds exact terms such as "Annex III" or "Article 113" that the embedding can miss. It uses the version chosen in step 5.

In [ ]:
from rank_bm25 import BM25Okapi
keep = lambda text: [w for w in words(text) if w not in REMOVE[CHOSEN]]
bm25 = BM25Okapi([keep(c["embed_text"]) for c in chunks])
print("BM25 index over", len(chunks), "chunks")

## 9. Search: dense (E1) and hybrid (E2)
- **Dense (E1, the baseline):** the closest vectors in Chroma.
- **Hybrid (E2):** the dense list and the BM25 list joined with Reciprocal Rank Fusion: each chunk gets `1 / (60 + rank)` from each list, and the two are added. 60 is the value from the RRF paper (Cormack et al., 2009).

Both return the top 5 (`top_k`); which one we keep, and whether 5 is right, is measured on the `val` questions in KAN-99.

In [ ]:
R = CONFIG["retrieval"]

def dense(question):
    q = embed([CONFIG["embedding"]["query_instruction"] + question])[0]
    return col.query(query_embeddings=[q.tolist()], n_results=R["candidates"])["ids"][0]

def keyword(question):
    return [chunks[n]["chunk_id"] for n in np.argsort(-bm25.get_scores(keep(question)))[:R["candidates"]]]

def hybrid(question):
    rrf = Counter()
    for ranked in (dense(question), keyword(question)):
        for rank, cid in enumerate(ranked, 1):
            rrf[cid] += 1 / (R["rrf_k"] + rank)
    return [cid for cid, _ in rrf.most_common(R["top_k"])]

[ref_of[c] for c in hybrid("Is an AI tool that grades students' essays high-risk?")]

**Comment:** _(write 1–2 sentences after running)_

## 10. A handful of test questions (11/11 check)
Five questions from `docs/eval/eval-set-v1.csv`, from Yeşim's (BM), Yaren's (EM) and my (B) questions. Next to the top 5 chunks of dense and hybrid search is the **gold source** the team wrote; ✓ = a gold provision.

In [ ]:
def show(ids, gold):
    parts = [g.strip() for g in gold.split(";")]
    return "\n".join(("✓ " if any(ref_of[c].startswith(g) or g.startswith(ref_of[c]) for g in parts) else "· ") + ref_of[c] for c in ids)

rows = [{"id": q, "question": questions[q]["question"], "dense (E1)": show(dense(questions[q]["question"])[:5], questions[q]["gold_source"]),
         "hybrid (E2)": show(hybrid(questions[q]["question"]), questions[q]["gold_source"]), "gold source": questions[q]["gold_source"]}
        for q in ["BM-01", "BM-07", "EM-09", "B-03", "B-04"]]
pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

**Comment:** _(write 1–2 sentences after running)_

## 11. A picture of the chunks
As in my earlier retrieval project, UMAP draws the 768 numbers of each chunk on a page. Chunks with a similar meaning should be close; each colour is one area of our provision list.

In [ ]:
import umap, matplotlib.pyplot as plt
xy = umap.UMAP(n_neighbors=10, random_state=42).fit_transform(vectors)
for area in sorted({c["area"] for c in chunks}):
    pts = xy[[c["area"] == area for c in chunks]]
    plt.scatter(pts[:, 0], pts[:, 1], s=25, label=area)
plt.legend(fontsize=7); plt.title("105 chunks, coloured by area"); plt.axis("off"); plt.show()

**Comment:** _(write 1–2 sentences after running)_

## Next: KAN-99
- Split the questions into `val` and `test` (KAN-53, 11/11).
- Repeat step 5 (removing words or not) on all the team's `val` questions.
- On every `val` question: Recall@5 and MRR (instructor's slides: *"compute Recall@k / MRR on gold IDs"*), dense (E1) vs hybrid (E2), and top-k 3, 5 or 8.
- The instructor's RAG metrics: Context Entities Recall (*"for fact-heavy text such as regulations"*) and, with the generated answers, Faithfulness.
- Everything logged in W&B; the baseline experiment card is KAN-100.